# Task 15 (server) — reproduce concept training before extending it

Server twin of `research_tasks_15_zhang_reproduction.ipynb` (Colab). Same
objective, same schedule, same evaluators; the difference is that the filesystem
persists, so nothing round-trips through Drive.

**How to run it.** Upload this one `.ipynb` to the server, open it, read the GPU
table in §1, edit the single control cell in §2 — GPU, model, HF token, which
stages run — and Run All. Nothing else is uploaded and no path is edited: the
notebook clones both repositories itself and keeps every artefact under its own
directory, so moving the file moves the whole experiment.

**One model per pass.** Every artefact is keyed on the model tag and the resume
logic reads finished work back by path, so `MODEL` in the control cell selects
the pass instead of a loop over models. Qwen3-1.7B-Base first; then change that
one line to `Qwen/Qwen3-4B-Base`, which reuses 1.7B's content words and skips the spaCy
POS pass entirely — the two tokenizers are compared before the copy and the run
aborts if they differ. Running them in parallel on two cards is also fine; 4B
then simply pays that pass itself, so either order is safe.

**Interruptions are cheap.** Extraction shards, trained adapters and finished
evaluation tables are all skipped on a re-run. If the kernel dies, reopen the
notebook and Run All: it picks up at the shard or arm that was in flight.

### Headless alternative, for an unattended pass

A kernel survives a dropped VPN — it runs on the server, not in the browser —
but the output stream does not always reattach, so a multi-hour pass is easier to
follow as a log file under `tmux`:

```bash
export CONCEPT_BASE=$PWD
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 \
    --output executed_qwen17.ipynb reproducibilty_15.ipynb 2>&1 | tee qwen17.log
```

The control cell's values win over the shell, so set one to `None` there to pass
it in from outside instead — which is how two models share the machine on
separate cards:

```bash
CONCEPT_MODEL=Qwen/Qwen3-1.7B-Base GPU_ID=1 jupyter nbconvert ... &
CONCEPT_MODEL=Qwen/Qwen3-4B-Base   GPU_ID=2 jupyter nbconvert ... &
```

## Layout

Everything — both clones, the HF cache, the corpus, the adapters and every
report — lands under the directory this notebook is sitting in. Nothing is
written to your home directory, and nothing outside this tree is touched:

```
<the directory holding this notebook>/
├── reproducibilty_15.ipynb
├── concept_aware/
│   ├── concept-aware-training/   our repo, cloned and pulled: patch + evaluators
│   ├── learning-concepts/        upstream, reset to the pinned commit, patched
│   ├── hf_cache/                 HF_HOME: model weights and the C4 shards
│   ├── data/                     extracted concept sets and the splits
│   └── runs/                     QLoRA adapters (small at r=4, kept for resume)
└── outputs/                      <- the only directory you copy back
    ├── qwen3-1.7b-base/
    │   ├── data_audit.json
    │   ├── results/       flat_main_table.csv, sts_*.csv, *_ci_*.json, mteb_raw/
    │   ├── logs/          per-arm training_history.jsonl
    │   └── run_manifests/ label -> adapter path, extraction shard completion
    ├── qwen3-4b-base/     same shape
    └── shared/            pip freeze, benchmark integrity (model-independent)
```

`outputs/` holds reports only — an assertion fails the run if model weights or
optimizer state ever reach it — so it stays small enough to `rsync` back. It is
also the layout the local analysis copy uses, so a finished model directory is
copied off the server as-is with no renaming.

The base directory is the notebook's own, found through `JPY_SESSION_NAME`, so a
kernel started somewhere else cannot scatter a second tree. Setting
`CONCEPT_BASE` overrides it.

## Before the long run

The control cell in §2 is the whole configuration; everything below it reads from
the environment it sets. Watch the **first extraction shard** anyway: on the A40
`SPACY_GPU` measured **5.0 s/sequence** against 42.6 on CPU, so at 5 s/seq the
4,000 sequences take about 5.5 h and at 40 they take days. A rate near 40 means
`CONCEPT_SPACY_GPU` did not reach the child process — the flag says GPU while
the work ran on CPU. (Missing cupy is the loud failure, not the slow one:
`spacy.require_gpu()` raises `No GPU devices detected` rather than falling back.)
Shards are recorded only once they finish, so interrupting after the first costs
nothing.

## 1. Which GPUs are free

Run this first, then name one in the control cell below. Getting it wrong means restarting the kernel, not just re-running a cell.

In [ ]:
# Which cards are free.  Pick one with spare memory and no other process, then
# name it in the control cell below -- BEFORE anything here touches CUDA.
!nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv

## 2. The one cell to edit

GPU, model, HF token and the stage gates. Everything after this reads them from the environment, including every child process.

In [ ]:
# ==== THE ONLY CELL YOU EDIT.  Set these, then Run All. ======================
import os

GPU_ID   = "1"                 # from the table above
MODEL    = "Qwen/Qwen3-1.7B-Base"   # ONE model per pass; then "Qwen/Qwen3-4B-Base".
                               # The -Base suffix is NOT cosmetic: a bare Qwen3 name is the
                               # post-trained chat model, and this study continues PRE-training
                               # and compares against Llama-3.2-1B, a base model.  (Qwen2.5 named
                               # them the other way round, which is how this gets picked wrong.)
HF_TOKEN = ""                  # gated models only (Llama).  Qwen3 is open: leave empty.
                               # If you do paste one, CLEAR IT BEFORE SAVING THIS FILE.

RUN_DATA    = True    # extract concept sets from C4 and build the splits (the long stage)
RUN_SMOKE   = True    # ~20-step objective check before any long training starts
RUN_SCREEN  = True    # train the seven arms at seed 42
RUN_CONFIRM = False   # retrain the headline arms across SEEDS; Qwen is a seed-42
                      # replication, so off.  It needs RUN_MULTISEED too -- on its own
                      # it loops over [42] alone and every arm is already finished.
RUN_MULTISEED = False # add seeds 123 and 2024 to SEEDS
RUN_EVAL    = True    # score every checkpoint: SWORDS, STS, perplexity, bm-semlex
SPACY_GPU   = True    # 8.5x faster extraction; needs cupy-cuda12x, installed below

# Everything below reads these from the environment, which is also how they reach
# each child process.  Set any value to None to defer to a variable exported in
# the shell instead -- that is what the nbconvert commands in the cell above use.
for _name, _value in {"GPU_ID": GPU_ID, "CONCEPT_MODEL": MODEL, "HF_TOKEN": HF_TOKEN or None,
                      "RUN_DATA": RUN_DATA, "RUN_SMOKE": RUN_SMOKE, "RUN_SCREEN": RUN_SCREEN,
                      "RUN_CONFIRM": RUN_CONFIRM, "RUN_EVAL": RUN_EVAL,
                      "RUN_MULTISEED": RUN_MULTISEED, "SPACY_GPU": SPACY_GPU}.items():
    if _value is not None:
        os.environ[_name] = ("1" if _value else "0") if isinstance(_value, bool) else str(_value)

# CUDA_VISIBLE_DEVICES has to be set before any torch import initialises the
# driver, and the install cell below imports torch to decide about torchao.  The
# setup cell sets it too, from GPU_ID -- but by then the choice can already be
# locked in, and the run would quietly land on card 0.
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.setdefault("GPU_ID", "1")
print("GPU", os.environ["CUDA_VISIBLE_DEVICES"],
      "|", os.environ.get("CONCEPT_MODEL", "(default)"),
      "| HF token", "set" if os.environ.get("HF_TOKEN") else "none",
      "| stages:", " ".join(stage for stage in ("RUN_DATA", "RUN_SMOKE", "RUN_SCREEN",
                                                "RUN_CONFIRM", "RUN_EVAL")
                            if os.environ.get(stage) == "1"))

## 3. Dependencies

Once per environment. The `transformers` pin matters: upstream's extractor reuses a prefix KV cache through an API removed in v5.

In [ ]:
# Install once per environment.  transformers is pinned LAST and BELOW 4.58 on
# purpose: upstream's extractor reuses a prefix KV cache through
# DynamicCache.from_legacy_cache, which transformers removed in v5.
%pip install -q accelerate peft bitsandbytes datasets spacy "mteb>=1.12" nltk scipy scikit-learn seaborn pandas pytest wandb
%pip install -q "transformers>=4.51,<4.58"
# torchao is a Colab-era fix: peft raises on torchao < 0.16 from inside
# PeftModel.from_pretrained.  But a CURRENT torchao evaluates torch.int1 at
# import, which torch < 2.6 does not define, and transformers imports torchao
# unconditionally whenever it is installed -- so on an older torch a mismatched
# torchao makes EVERY model unloadable, with the traceback pointing at the model
# class rather than at torchao.  Install it only where it helps; remove it
# otherwise, which is safe because peft only needs it when it is present.
import subprocess, sys, torch
_version = tuple(int(part) for part in torch.__version__.split(".")[:2])
if _version >= (2, 6):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16"], check=False)
    print("torchao: installed for torch", torch.__version__)
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=False)
    print("torchao: removed, torch", torch.__version__, "predates torch.int1")
# cupy backs spacy.require_gpu(); without it thinc raises and SPACY_GPU must be
# set False.  thinc reads cupy's presence at import, so install before spacy loads.
%pip install -q cupy-cuda12x
!python -m spacy download en_core_web_sm
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

In [ ]:
import os
# Set in the control cell above; re-applied here so this cell stands alone when
# it is re-run on its own, and so the headless path (GPU_ID=2 jupyter nbconvert
# --execute ...) works with the control cell's GPU_ID set to None.
GPU_ID = os.environ.get("GPU_ID", "1")
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from pathlib import Path
import datetime, hashlib, json, re, shutil, subprocess, sys, torch

# Every clone, dataset, checkpoint and result lives under BASE, so the whole
# experiment is one directory to archive or copy off the server.
def _notebook_base():
    """Directory the notebook itself lives in, so `outputs/` lands beside it.

    CONCEPT_BASE wins when set.  Otherwise prefer JPY_SESSION_NAME, which Jupyter
    sets to the notebook's own path: the working directory is the notebook's
    directory only when the kernel happened to start there, so `jupyter nbconvert
    --execute` invoked from anywhere else would scatter a second outputs/ tree
    next to wherever it was run from.
    """
    override = os.environ.get("CONCEPT_BASE")
    if override:
        return Path(override)
    session = os.environ.get("JPY_SESSION_NAME", "")
    if session.endswith(".ipynb") and Path(session).parent.is_dir():
        return Path(session).parent
    return Path.cwd()

BASE = _notebook_base().resolve()
WORK = BASE / "concept_aware"
MAIN = WORK / "concept-aware-training"     # our repo: patch, evaluators, scripts
EXT = WORK / "learning-concepts"           # upstream, pinned
DATA = WORK / "data"                       # CONCEPT_DATA_ROOT
RUNS = WORK / "runs"                       # adapters (small at r=4, kept)
OUTPUTS = BASE / "outputs"                 # everything you download for analysis

# ONE model per pass.  Every artefact below is keyed on the model tag and the
# resume logic reads finished work back by path, so the model is selected in the
# control cell rather than looped over here -- which is also what lets two models
# share the machine without sharing a GPU.
BASE_MODEL = os.environ.get("CONCEPT_MODEL", "Qwen/Qwen3-1.7B-Base")
# Every per-model artifact is keyed on this tag.  Two models must never share a
# path: adapters resume by path, so a collision hands one model's weights to
# another and the run still looks like it succeeded.  conceptlib.paths derives
# the same tag the same way, so the data directories line up with these.
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
# combined.jsonl depends only on the tokenizer, so a model sharing a vocabulary
# with one already extracted produces a byte-identical file.  Naming the donor
# skips the spaCy POS pass; the vocabularies are compared before the copy and the
# run aborts if they differ.  Each pair shares a tokenizer WITHIN its family and
# never across families, which is why this is a table and not a size heuristic.
VOCAB_DONOR = {"Qwen/Qwen2.5-3B": "Qwen/Qwen2.5-1.5B",
               "Qwen/Qwen3-4B-Base": "Qwen/Qwen3-1.7B-Base",
               "Qwen/Qwen3-4B": "Qwen/Qwen3-1.7B",
               "meta-llama/Llama-3.2-3B": "meta-llama/Llama-3.2-1B"}
REUSE_CONTENT_WORDS_FROM = os.environ.get("CONCEPT_REUSE_FROM") or VOCAB_DONOR.get(BASE_MODEL)

# outputs/<model tag>/{data_audit.json,results,logs,run_manifests}, plus one
# shared directory for the two model-independent reports.  Unlike the Colab
# notebook there is no legacy unsuffixed path to preserve here, so EVERY model
# gets its own directory -- including the first one.
MODEL_OUT = OUTPUTS / MODEL_TAG
RESULT_DIR = MODEL_OUT / "results"
LOG_DIR = MODEL_OUT / "logs"
SHARED = OUTPUTS / "shared"
DATA_AUDIT = MODEL_OUT / "data_audit.json"
PRIMARY_SEED = 42
# One seed screens the pipeline and shows the direction of every effect, but it
# CANNOT support a claim: the pre-registered rule needs all three to agree in
# sign.  Flip to True for the reportable run; finished arms are skipped.
def _flag(name, default):
    """Read a run flag from the environment, defaulting to the value here.

    The control cell writes the flags into the environment rather than binding
    them here, so the same file also runs headless -- `CONCEPT_MODEL=...
    RUN_DATA=1 jupyter nbconvert --execute` drives one stage under tmux, which
    survives a dropped VPN, and the notebook stays reproducible either way.
    """
    return os.environ.get(name, str(default)).strip().lower() in ("1", "true", "yes", "on")

# Requires cupy (installed below).  Verified output-identical to CPU spaCy.
SPACY_GPU = _flag("SPACY_GPU", True)
RUN_MULTISEED = _flag("RUN_MULTISEED", False)
SEEDS = [PRIMARY_SEED] + ([123, 2024] if RUN_MULTISEED else [])
UPSTREAM_COMMIT = "b1d414143d11c8ed988b4cccbb06626cc8272bbe"

# Capture an existing `huggingface-cli login` BEFORE redirecting HF_HOME.  The
# token lives under the DEFAULT HF_HOME, so once we move HF_HOME into the project
# directory a freshly spawned child finds no token and gated downloads 401 --
# even though the parent, which imported huggingface_hub earlier, looks fine.
# Exporting HF_TOKEN makes auth explicit and inherited by every subprocess.  A
# token pasted into the control cell is already in the environment and wins here.
_cli_token = Path.home() / ".cache" / "huggingface" / "token"
if not os.environ.get("HF_TOKEN") and _cli_token.is_file():
    os.environ["HF_TOKEN"] = _cli_token.read_text().strip()
os.environ["HF_HOME"] = str(WORK / "hf_cache")

# Defaults for the Qwen3-1.7B-Base server pass; the control cell sets all of them, so
# these apply only when a flag is left None there or this cell is re-run alone.
# The Colab notebook keeps them False because a stray Run All there costs money
# and a session slot.  Here the whole point is an unattended pass, and every
# stage is resumable, so an accidental start costs the shard in flight.
RUN_DATA = _flag("RUN_DATA", True)
RUN_SMOKE = _flag("RUN_SMOKE", True)
RUN_SCREEN = _flag("RUN_SCREEN", True)
# Three seeds are a Colab job for the headline arms only; Qwen is a second-family
# replication at seed 42, so this stays off.
RUN_CONFIRM = _flag("RUN_CONFIRM", False)
# Task 15b only, and only after the 1B gate has chosen an arm.
RUN_HYBRID = _flag("RUN_HYBRID", False)
RUN_NEGATIVE_CONTROLS = _flag("RUN_NEGATIVE_CONTROLS", False)
RUN_EVAL = _flag("RUN_EVAL", True)
# Skip any run whose artefacts already exist.  A killed job resumes from here.
RESUME_FINISHED_RUNS = True

# Extraction precision.  Upstream inherits use_4bit=True from TrainingConfig,
# where it exists for QLoRA TRAINING.  Extraction runs ~94 small forwards per
# sequence; measured on an L4, bf16 was ~25% faster and avoids quantisation
# noise in the top-100 pool and the 0.75 cosine threshold the method depends on.
EXTRACT_4BIT = False
# Must match the Colab notebook: the two variants feed one study, and a model
# extracted here on 10,000 sequences could not be compared with one extracted
# there on 4,000.  Zhang et al. Fig. 8 reports STS unchanged at a quarter of the
# data; 4,000 keeps the 80/10/10 ratio.  Raise BOTH to 10000 for the strict
# reproduction.  merge_synonym_parts hard-fails unless the split sizes sum to the
# rows the shards actually cover.
EXTRACT_SEQUENCES = 4000
# Sequences per extraction shard, and so exactly what a killed job costs: a shard
# is only recorded as finished once it completes.
EXTRACT_SHARD = 500
SPLIT_TRAIN = int(EXTRACT_SEQUENCES * 0.8)
SPLIT_VAL = SPLIT_TEST = int(EXTRACT_SEQUENCES * 0.1)

_BAR = re.compile(r"\b(\d+)/(\d+)\s*\[")     # bounded tqdm: "  200/1000 ["
_BAR_OPEN = re.compile(r"\b(\d+)it\s*\[")     # unbounded tqdm: "  3200it [00:49"
PROGRESS_EVERY = 100

def run(argv, cwd=None, env=None):
    """Run a child process, streaming its output and keeping the tail on failure."""
    argv = list(map(str, argv))
    print("+", " ".join(argv), flush=True)
    merged = os.environ.copy()
    merged.update({"CONCEPT_DATA_ROOT": str(DATA),
                   "CONCEPT_CHECKPOINT_ROOT": str(RUNS),
                   "CONCEPT_RESULTS_ROOT": str(OUTPUTS),
                   # The POS filter is ~90% of extraction; on GPU it ran 8.5x
                   # faster on an A40 (42.6 -> 5.0 s/seq) with 12400/12400 POS
                   # tags identical to CPU.  SPACY_GPU gates it because it needs
                   # cupy, and because the Colab-produced Llama data did not use it.
                   "CONCEPT_SPACY_GPU": "1" if SPACY_GPU else "0",
                   # This machine has no locale set, so Python defaults to ASCII
                   # and both the child and its captured output die on C4's
                   # non-ASCII text, partway through a long run.
                   "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"})
    if env: merged.update(env)
    process = subprocess.Popen(argv, cwd=cwd, env=merged, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        line = line.replace("\r", "")
        tail.append(line)
        del tail[:-40]
        hit = _BAR.search(line)
        if hit:
            done, total = int(hit.group(1)), int(hit.group(2))
            if done % PROGRESS_EVERY and done != total:
                continue
        else:
            # Dataset loading has no total ("3200it [00:49"), so there is no final
            # count to anchor on.  Thin it ten times harder; it is pure noise and
            # a full sweep would otherwise emit tens of thousands of lines.
            loose = _BAR_OPEN.search(line)
            if loose and int(loose.group(1)) % (PROGRESS_EVERY * 10):
                continue
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(
            f"command failed with exit code {code}\n  {' '.join(argv)}\n"
            f"--- last {len(tail)} lines of its output ---\n{''.join(tail)}")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def sync_small_artifacts(source, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for path in Path(source).rglob("*"):
        if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png", ".log"}:
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

def audit_outputs():
    """Keep OUTPUTS downloadable: reports only, no multi-GB weights."""
    forbidden = {"pytorch_model.bin", "model.safetensors", "optimizer.pt",
                 "scheduler.pt", "scaler.pt", "rng_state.pth"}
    found = [str(p) for p in OUTPUTS.rglob("*")
             if p.name in forbidden or p.name.startswith("checkpoint-")]
    assert not found, f"full-weight/optimizer artifacts reached OUTPUTS: {found}"

# The server filesystem persists, so the Colab Drive round-trip is unnecessary.
# These keep the same names and call sites as the Colab notebook, which is what
# lets the two share every experiment cell below.
def cache_dataset(leaf): pass

def restore_dataset(leaf):
    return (Path(leaf) / "synonyms_train.jsonl").is_file()

def cache_adapter(path): return Path(path)

def restore_adapter(path):
    return (Path(path) / "adapter_config.json").is_file()

# Inside THIS model's output directory.  A manifest shared between models would
# name another model's adapters, and restore_all would load them without
# complaint -- the resume path has no way to tell whose weights it just read.
RUN_MANIFEST = MODEL_OUT / "run_manifests"

def save_runs(runs, name):
    RUN_MANIFEST.mkdir(parents=True, exist_ok=True)
    (RUN_MANIFEST / f"{name}.json").write_text(
        json.dumps({k: str(v) for k, v in runs.items()}, indent=2))

def load_runs(name):
    path = RUN_MANIFEST / f"{name}.json"
    return {} if not path.is_file() else {k: Path(v) for k, v in json.loads(path.read_text()).items()}

def restore_all(runs):
    live = {}
    for label, path in runs.items():
        if restore_adapter(path):
            live[label] = Path(path)
        else:
            print("missing adapter, dropping from this pass:", label)
    return live

def eval_done(marker):
    return Path(marker).is_file() and RESUME_FINISHED_RUNS

def eval_covered(path, checkpoints):
    """True when `path` already scores every checkpoint of THIS pass.

    Coverage, not mere existence: adding a seed grows `checkpoints`, the old file
    stops covering it, and the evaluator reruns.  A plain existence check would
    report the previous pass's table as if it were this one's.
    """
    if not (RESUME_FINISHED_RUNS and Path(path).is_file()):
        return False
    try:
        rows = json.loads(Path(path).read_text())
    except (json.JSONDecodeError, OSError):
        return False              # truncated by a killed job mid-write; redo it
    if not isinstance(rows, list):
        return False
    scored = {str(row.get("checkpoint")) for row in rows if isinstance(row, dict)}
    return set(map(str, checkpoints)) <= scored

def sts_covered(path):
    """True when one STS pass already wrote its nine task rows to `path`."""
    if not (RESUME_FINISHED_RUNS and Path(path).is_file()):
        return False
    with open(path, encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip()) >= 10   # header + 9 tasks

def guarded(path, checkpoints, argv, cwd, what):
    """Run one evaluator unless its output already covers every checkpoint."""
    if eval_covered(path, checkpoints):
        print(f"resume: {what} already covers {len(checkpoints)} checkpoints, skipping")
        return
    run(argv, cwd=cwd)

def assert_under_base(path):
    resolved = Path(path).resolve()
    assert str(resolved).startswith(str(BASE)), f"{resolved} escapes {BASE}"

for directory in (WORK, DATA, RUNS, OUTPUTS, MODEL_OUT, RESULT_DIR, LOG_DIR, SHARED):
    directory.mkdir(parents=True, exist_ok=True)
print("BASE      ", BASE)
print("MODEL     ", BASE_MODEL, "->", MODEL_TAG)
print("REUSE FROM", REUSE_CONTENT_WORDS_FROM or "(nothing: full extraction)")
print("OUTPUTS   ", MODEL_OUT)
print("GPU       ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# Both clones are required: MAIN carries the patch, the evaluators and the
# summary script; EXT is the pinned upstream the patch applies to.
if not MAIN.exists():
    run(["git", "clone", "https://github.com/SharvaGogawale1/concept-aware-training.git", MAIN])
else:
    run(["git", "-C", str(MAIN), "pull", "--ff-only"])
if not EXT.exists():
    run(["git", "clone", "https://github.com/christine-zhang1/learning-concepts.git", EXT])
# Reset to the pinned commit and wipe every patch artefact before re-applying.
# Testing "does it apply, else does it reverse-apply" only worked while the patch
# never changed: once MAIN pulls a newer one, the old patch is applied, neither
# direction matches, and the run dies on an assertion.  Resetting is idempotent
# and always ends in the same state.  EXT holds upstream code only -- the corpus
# lives in DATA, outside it -- so clean -fd is safe.
run(["git", "-C", str(EXT), "reset", "--hard", UPSTREAM_COMMIT])
run(["git", "-C", str(EXT), "clean", "-fdq"])

patch_file = MAIN / "external" / "learning-concepts.patch"
assert patch_file.exists(), f"{patch_file} missing; push it before running here."
run(["git", "apply", str(patch_file)], cwd=EXT)
print("patch applied onto", UPSTREAM_COMMIT[:7])

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(EXT), "--no-deps"])

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

run([sys.executable, MAIN / "builddataset/verify_task14_data.py",
     "--repo_root", MAIN, "--download_missing",
     "--report_json", SHARED / "external_benchmark_integrity.json"], cwd=MAIN)
(SHARED / "environment_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))

# Prove the model is reachable FROM A SUBPROCESS, which is where every download
# actually happens.  The parent can hold credentials a freshly spawned child does
# not inherit, and the failure then surfaces 20 minutes later as a bare 401 on
# config.json whose traceback says nothing about authentication.
# The token is optional because gating is: Qwen2.5 is openly licensed, Llama-3.2
# is gated.  Demanding a token unconditionally would block a Qwen-only run for no
# reason, so let the reachability probe be the thing that decides.
from huggingface_hub import login, whoami
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face:", whoami()["name"])
else:
    print("No HF_TOKEN set; continuing on the assumption this model is ungated.")
_probe = subprocess.run(
    [sys.executable, "-c",
     "import sys; from transformers import AutoConfig;"
     "AutoConfig.from_pretrained(sys.argv[1]);"
     "print('model repo reachable from a subprocess')", BASE_MODEL],
    env={**os.environ}, capture_output=True, text=True)
assert _probe.returncode == 0, (
    f"{BASE_MODEL} is not reachable from a child process.  If it is a gated repo, "
    "run `huggingface-cli login` on this machine or export HF_TOKEN before "
    f"starting Jupyter, then re-run this cell.\n{_probe.stderr[-2000:]}")
print(_probe.stdout.strip())

## Rebuild and audit the concept data for this model

The audit hard-fails on split overlap, target misalignment, empty sets, or any concept that is not a complete single token. The observed target is part of every set by construction.


In [ ]:
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
# A cheap existence check here; kept unconditional so the body stays identical
# to the Colab notebook's, where it is a real restore from Drive.
restore_dataset(LEAF)
if RUN_DATA:
    # The extraction is the longest stage.  A shard is recorded as finished only
    # once it completes, so a killed job costs at most the shard in progress.
    # get_content_words.py streams into combined.jsonl, so an interrupted pass
    # leaves a SHORT file behind and existence alone does not mean completion.
    # Test for "enough rows", not "exactly EXTRACT_SEQUENCES": MAX_SAMPLES is
    # hardcoded to 10000 in that script, so the file legitimately holds 10000 rows
    # even when we only extract the first 4000, and an equality test would delete
    # and regenerate it on every resume.
    combined = LEAF.parent / "combined.jsonl"
    if not combined.is_file() and REUSE_CONTENT_WORDS_FROM:
        donor_tag = REUSE_CONTENT_WORDS_FROM.split("/")[-1].lower()
        donor = DATA / "c4" / donor_tag / "combined.jsonl"
        if not donor.is_file():
            restore_dataset(DATA / "c4" / donor_tag / "embedding")
        # The donor is a shortcut, never a requirement.  If it has not been
        # extracted yet, fall through and generate this model's own content words
        # rather than dying in shutil.copy2 -- which is also what lets two models
        # of one family run concurrently on separate GPUs: whichever starts first
        # simply pays the POS pass itself.
        if not donor.is_file():
            print("donor", donor, "not extracted yet; generating content words here")
        else:
            from transformers import AutoTokenizer
            assert (AutoTokenizer.from_pretrained(BASE_MODEL).get_vocab()
                    == AutoTokenizer.from_pretrained(REUSE_CONTENT_WORDS_FROM).get_vocab()), (
                f"{BASE_MODEL} and {REUSE_CONTENT_WORDS_FROM} do not share a vocabulary, "
                "so their content words differ and must be regenerated")
            combined.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(donor, combined)
            print("reused content words from", REUSE_CONTENT_WORDS_FROM)
    if combined.is_file():
        rows = sum(1 for _ in combined.open())
        if rows < EXTRACT_SEQUENCES:
            print(f"combined.jsonl has {rows} rows, need {EXTRACT_SEQUENCES}; regenerating")
            combined.unlink()
        else:
            print(f"combined.jsonl has {rows} rows, using the first {EXTRACT_SEQUENCES}")
    if not combined.is_file():
        run([sys.executable, "data/get_content_words.py", "--model", BASE_MODEL,
             "--dataset", "c4", "--max_length", "256"], cwd=EXT)
        cache_dataset(LEAF)
    # An interrupted shard leaves PARTIAL synonyms_/topk_ files behind, so their
    # mere existence does not mean the shard finished.  Record completion in a
    # manifest on disk instead; embedding_synonyms.py truncates both files when
    # it restarts a shard, so a re-run is always clean.
    # The manifest lives in this model's own output directory, so one model's
    # finished shards can never mark another's as done and skip extraction.
    shards_done = load_runs("task15_shards")
    for start in range(0, EXTRACT_SEQUENCES, EXTRACT_SHARD):
        end = start + EXTRACT_SHARD
        key = f"{start}_{end}"
        synonym_part = LEAF / f"synonyms_{start}_{end}.jsonl"
        topk_part = LEAF.parent / "prompting" / f"topk_{start}_{end}.jsonl"
        if (RESUME_FINISHED_RUNS and key in shards_done
                and synonym_part.is_file() and topk_part.is_file()):
            print("resume: extraction shard already complete", start, end)
            continue
        run([sys.executable, "data/embedding_synonyms.py", "c4",
             "--start", start, "--end", end, "--model", BASE_MODEL,
             *([] if EXTRACT_4BIT else ["--no-4bit"])], cwd=EXT)
        cache_dataset(LEAF)
        shards_done[key] = synonym_part
        save_runs(shards_done, "task15_shards")
    run([sys.executable, "data/merge_synonym_parts.py", "--train-size", SPLIT_TRAIN,
         "--val-size", SPLIT_VAL, "--test-size", SPLIT_TEST, "--expected-count", "2", "--force"], cwd=EXT)
    run([sys.executable, "data/augment_synonyms.py", "--base-dir", DATA,
         "--num-augmentations", "4", "--seed", "42", "--overwrite"], cwd=EXT)
    run([sys.executable, "data/randomize_synonyms.py", "--split", "train", "--overwrite"], cwd=EXT)

if RUN_DATA:
    cache_dataset(LEAF)
if RUN_DATA:
    run([sys.executable, "data/audit_concept_data.py",
         "--train", LEAF / "synonyms_train.jsonl",
         "--validation", LEAF / "synonyms_val.jsonl",
         "--test", LEAF / "synonyms_test.jsonl",
         "--tokenizer", BASE_MODEL,
         "--expected-train", SPLIT_TRAIN, "--expected-validation", SPLIT_VAL,
         "--expected-test", SPLIT_TEST,
         # Inside this model's directory: a shared path means the second model's
         # audit silently overwrites the first's, and the provenance of the
         # finished data is exactly what an audit report exists to preserve.
         "--report", DATA_AUDIT], cwd=EXT)

## Unit tests and eight-row GPU smoke run

The smoke run checks the complete QLoRA path before any sweep. It is deleted immediately. The tests cover the released loss, our optional objectives, gradients, and hierarchy sequence scoring.


In [ ]:
if RUN_SMOKE:
    run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=EXT)
    # No map-style preprocessing cache is used. Two independent loads must
    # still produce identical candidate supervision.
    # pip install -e EXT only installs the conceptlib PACKAGE (all pyproject
    # declares); train.py is a loose top-level module, so importing it in-process
    # needs EXT on sys.path.  The subprocess calls are unaffected -- they pass
    # cwd=EXT -- which is why this only bites the in-notebook import.
    if str(EXT) not in sys.path:
        sys.path.insert(0, str(EXT))
    from train import ConceptDataset
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    first = ConceptDataset(LEAF / "synonyms_train.jsonl", tokenizer, max_samples=8)
    second = ConceptDataset(LEAF / "synonyms_train.jsonl", tokenizer, max_samples=8)
    assert [x["content_words"] for x in first.data] == [x["content_words"] for x in second.data]
    smoke = RUNS / "smoke"
    assert_under_base(smoke)
    run([sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
         "--dataset-type", "embedding", "--concept-loss-weight", "1.0",
         "--max-train-samples", "8", "--num-train-epochs", "1",
         "--output-dir", smoke, "--save-strategy", "no", "--report-to", "none"], cwd=EXT)
    assert (smoke / "adapter_config.json").exists()
    # Required one-model equivalence test: adapter logits and merged logits.
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    adapted = PeftModel.from_pretrained(base, smoke).eval()
    probe = tokenizer("A dog is an animal.", return_tensors="pt")
    with torch.no_grad(): adapter_logits = adapted(**probe).logits
    merged = adapted.merge_and_unload().eval()
    with torch.no_grad(): merged_logits = merged(**probe).logits
    assert torch.allclose(adapter_logits, merged_logits, atol=2e-4, rtol=2e-4)
    del merged, adapted, base
    shutil.rmtree(smoke)

## Exact reproduction schedule

Seed 42 gets the complete $\lambda\in\{0.25,0.5,0.75,1\}$ curve. The headline NTP, one-epoch augmented NTP, randomized $\lambda=.25$, and concept-marginal $\lambda=1$ settings are then confirmed with seeds 42, 123, and 2024. Released effective batch size is logged. If the headline fails, only seed 42 is rerun with the paper-stated batch before any diagnosis.


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
# The server filesystem persists, so this is a cheap existence check.
restore_dataset(LEAF)

def adapter_path(method, seed, value):
    path = RUNS / MODEL_TAG / method / f"seed_{seed}" / str(value)
    assert_under_base(path)
    return path

def finished(path):
    """A run counts as finished when its adapter is on disk."""
    if (Path(path) / "adapter_config.json").is_file():
        return True
    return restore_adapter(path)

def train_flat(method, seed, concept_weight, *, objective="set_marginal",
               slot_ntp_weight=None, contrast_beta=0.0, exclude_target=False,
               randomized=False, data_augmentation=False, epochs=5, train_file=None,
               max_samples=None, batch=8, accum=2, alt_aux="none", alt_aux_weight=0.0):
    # The aux suffix is added only when the term is on, so every adapter trained
    # before it existed keeps its path and still resumes.
    aux_tag = "" if alt_aux == "none" else f"_aux_{alt_aux}_{alt_aux_weight}"
    out = adapter_path(method, seed, f"lambda_{concept_weight}_beta_{contrast_beta}{aux_tag}")
    args = [sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
            "--dataset-type", "embedding", "--concept-loss-weight", concept_weight,
            "--concept-objective", objective, "--contrast-beta", contrast_beta,
            "--seed", seed, "--num-train-epochs", epochs, "--output-dir", out,
            "--save-strategy", "no", "--report-to", "none",
            "--per-device-train-batch-size", batch,
            "--gradient-accumulation-steps", accum]
    if slot_ntp_weight is not None: args += ["--slot-ntp-weight", slot_ntp_weight]
    if alt_aux != "none": args += ["--alt-aux", alt_aux, "--alt-aux-weight", alt_aux_weight]
    # Drops the observed target from the concept set, so the loss cannot be paid
    # with the mass NTP already put there.  Pair it with slot_ntp_weight=1.0 or
    # the observed target is pushed down.  adapter_path() does not encode this
    # flag, so the METHOD name must differ from the target-inclusive arm's.
    if exclude_target: args += ["--exclude-target"]
    if randomized: args += ["--randomized-synonyms"]
    if data_augmentation: args += ["--use-data-augmentation"]
    if train_file: args += ["--train-file", train_file]
    if max_samples: args += ["--max-train-samples", max_samples]
    # Released effective batch is 8 x 2 = 16; it is recorded in every config.
    if RESUME_FINISHED_RUNS and finished(out):
        print("resume: already trained, skipping", out)
        return out
    run(args, cwd=EXT)
    cache_adapter(out)
    return out

In [ ]:
REPRO_RUNS = load_runs("task15")
if RUN_SCREEN:
    REPRO_RUNS["ntp_seed42"] = train_flat("ntp", 42, 0.0)
    for lam in [0.25, 0.5, 0.75, 1.0]:
        REPRO_RUNS[f"zhang_lambda{lam}_seed42"] = train_flat("zhang_marginal", 42, lam)
    REPRO_RUNS["augmented_ntp_seed42"] = train_flat("augmented_ntp", 42, 0.0, epochs=1,
        train_file=LEAF / "synonyms_train_aug5x.jsonl", data_augmentation=True)
    REPRO_RUNS["randomized_seed42"] = train_flat("randomized", 42, 0.25, randomized=True)

if RUN_CONFIRM:
    for seed in SEEDS:
        REPRO_RUNS[f"ntp_seed{seed}"] = train_flat("ntp", seed, 0.0)
        REPRO_RUNS[f"augmented_ntp_seed{seed}"] = train_flat("augmented_ntp", seed, 0.0,
            epochs=1, train_file=LEAF / "synonyms_train_aug5x.jsonl", data_augmentation=True)
        REPRO_RUNS[f"randomized_seed{seed}"] = train_flat("randomized", seed, 0.25, randomized=True)
        # Zhang runs the semantic control at lambda=0.25 while the headline
        # concept arm runs at lambda=1.0.  Any claim that concept training does
        # or does not beat the control needs it at the SAME weight, otherwise the
        # comparison confounds the candidate sets with the mixing weight.
        REPRO_RUNS[f"randomized_lambda1.0_seed{seed}"] = train_flat(
            "randomized", seed, 1.0, randomized=True)
        REPRO_RUNS[f"zhang_seed{seed}"] = train_flat("zhang_marginal", seed, 1.0)
    # The lambda sweep already trained zhang_marginal at lambda=1.0 on seed 42, and
    # adapter_path() maps both calls to the same directory.  Two dict keys pointing at
    # one adapter would score it twice and report it as two arms.
    REPRO_RUNS.pop("zhang_lambda1.0_seed42", None)
save_runs(REPRO_RUNS, "task15")

# Use only if the released-batch seed-42 reproduction misses the stated trend.
# This is a named sensitivity run, never a replacement or a cherry-picked row.
RERUN_PAPER_STATED_BATCH = False
if RERUN_PAPER_STATED_BATCH:
    REPRO_RUNS["zhang_seed42_paper_batch"] = train_flat(
        "zhang_marginal_paper_batch", 42, 1.0, batch=2, accum=1)

## Evaluation and learning curves

Every checkpoint is scored by the same evaluators. “Global NLL” covers every next-token position; “content-word NLL” covers the semantic slots; “set mass” is total probability assigned to the target-inclusive valid set. SWORDS and bm-semlex are zero-shot here.


In [ ]:
if RUN_EVAL:
    # A fresh session has the manifest but not the weights; pull them back first.
    REPRO_RUNS = restore_all(REPRO_RUNS)
    checkpoints = [BASE_MODEL, *map(str, REPRO_RUNS.values())]
    result_dir = RESULT_DIR
    result_dir.mkdir(parents=True, exist_ok=True)
    # Each evaluator is skipped only when its own output already scores every
    # checkpoint of this pass, so a disconnect costs at most one evaluator
    # instead of the whole section.
    guarded(result_dir / "perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "perplexity.json"], EXT, "perplexity")
    guarded(result_dir / "concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "concept_sets.json"], EXT, "concept sets")
    for label, checkpoint in {"pretrained": BASE_MODEL, **REPRO_RUNS}.items():
        display_label = label.replace("_", " ")
        csv_output = result_dir / f"sts_{display_label}.csv"
        # STS writes one CSV per checkpoint, so it resumes per checkpoint.
        if sts_covered(csv_output):
            print("resume: STS already scored, skipping", display_label)
            continue
        mteb_args = [sys.executable, "eval/eval_mteb.py", "--base-model", BASE_MODEL,
                     "--dataset", "c4", "--dataset-type", "embedding", "--tasks", "sts",
                     "--run-label", display_label,
                     "--csv-output", csv_output,
                     "--mteb-output-root", result_dir / "mteb_raw"]
        mteb_args += ["--no-adapter"] if checkpoint == BASE_MODEL else ["--adapter-path", checkpoint]
        run(mteb_args, cwd=EXT)
    guarded(result_dir / "swords_zero_shot.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--swords_json", MAIN / "data/swords/swords-v1.1_dev.json.gz",
             "--results_json", result_dir / "swords_zero_shot.json", "--modes", "left", "full"],
            MAIN, "SWORDS")
    # Seconds to run, and each must always match the SWORDS file it reads, so
    # these are never skipped.  Pretrained is the reference row; NTP and
    # augmented NTP are the matched controls the causal claims are stated
    # against.  The index is looked up by label rather than hardcoded: RUN_CONFIRM
    # pops zhang_lambda1.0_seed42, so positions shift once seeds are added and a
    # literal index would quietly compare against the wrong arm.
    # randomized is the one that decides whether the SWORDS gain is semantic:
    # it is trained with the same objective on meaningless candidate sets, so a
    # concept arm that does not separate from it there is buying its GAP with
    # distributional smoothing rather than with concept content.
    baselines = {"pretrained": BASE_MODEL}
    for label in ("ntp_seed42", "augmented_ntp_seed42", "randomized_seed42"):
        if label in REPRO_RUNS:
            baselines[label] = str(REPRO_RUNS[label])
    for label, reference in baselines.items():
        run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
             "--kind", "swords", "--results-json", result_dir / "swords_zero_shot.json",
             "--baseline-index", checkpoints.index(reference),
             "--output", result_dir / f"swords_paired_ci_vs_{label}.json"], cwd=MAIN)
    guarded(result_dir / "bm_semlex.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_bm_semlex.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--data", MAIN / "data/bm_semlex/curated_200.tsv",
             "--results_json", result_dir / "bm_semlex.json"], MAIN, "bm-semlex")
    manifest = {"pretrained": BASE_MODEL, **{label.replace("_", " "): str(path) for label, path in REPRO_RUNS.items()}}
    (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    run([sys.executable, MAIN / "scripts/summarize_concept_experiments.py",
         "--manifest", result_dir / "manifest.json", "--result-dir", result_dir,
         "--output", result_dir / "flat_main_table.csv"], cwd=MAIN)
    for label, path in REPRO_RUNS.items(): sync_small_artifacts(path, LOG_DIR / label)
    audit_outputs()

In [ ]:
# Plot only scalar learning curves; adapters stay under RUNS.
import pandas as pd
from matplotlib import pyplot as plt
curves = []
for label, path in REPRO_RUNS.items():
    history = Path(path) / "training_history.jsonl"
    if history.exists():
        frame = pd.read_json(history, lines=True)
        frame["method"] = label
        curves.append(frame)
if not curves:
    print("no training_history.jsonl found; nothing to plot")
else:
    frame = pd.concat(curves, ignore_index=True)
    # The trainer logs ce_loss and concept_loss once per EVAL BATCH, two or three
    # rows sharing one global_step.  Only the row that also carries eval_loss is
    # the aggregate over the validation set; plotting the rest draws intra-eval
    # scatter as if it were training dynamics.
    train_rows = frame.dropna(subset=["loss"])
    eval_rows = frame.dropna(subset=["eval_loss"])
    panels = [(train_rows, "step", "loss", "training loss (NOT comparable across arms:\n"
               "each minimises a different mix of the two terms)"),
              (eval_rows, "epoch", "ce_loss", "validation NTP cross-entropy"),
              (eval_rows, "epoch", "concept_loss", "validation concept loss")]
    # One fixed colour per method.  Letting matplotlib assign them per panel makes
    # the colours shift wherever an arm is dropped for having no concept loss,
    # while the legend sits on the first panel only -- so the same colour means
    # different arms in different panels.
    methods = sorted(frame["method"].unique())
    colours = dict(zip(methods, plt.cm.tab10.colors * (1 + len(methods) // 10)))
    figure, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for axis, (rows, x, y, title) in zip(axes, panels):
        if y not in rows:
            continue
        for label in methods:
            group = rows[rows["method"] == label].dropna(subset=[y])
            if not group.empty and group[y].abs().sum() > 0:
                axis.plot(group[x], group[y], marker="o", ms=3, lw=1.4,
                          color=colours[label], label=label)
        axis.set_xlabel(x); axis.set_title(title, fontsize=9); axis.grid(alpha=.25, lw=.5)
    axes[0].legend(fontsize=7, frameon=False, ncol=2)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / "training_curves.png", dpi=180)
    plt.show()

## Reproduction gate

Proceed only if Zhang concept marginal beats NTP and augmented NTP on mean STS, improves content-word perplexity over NTP, stays near pretrained global perplexity, and the $\lambda$ curve has the reported direction. If seed 42 fails, rerun that one setting with the paper-stated effective batch and report both configurations—do not silently substitute it.
